# Task 1: Fine-Tuning Qwen2.5-0.5B with LoRA for Structured Intent Extraction
### tndrr AI/ML Internship Technical Assessment — Applied LLM Systems
**Author:** Harsh Maurya | **GitHub:** [applied-llm-systems](https://github.com/Advik-harsha/applied-llm-systems)

This notebook demonstrates fine-tuning a small open model (`Qwen/Qwen2.5-0.5B-Instruct`, 490M parameters) using Low-Rank Adaptation (LoRA) on a single free Google Colab T4 GPU (~7 minutes runtime).

**Objective:** Convert messy, unstructured natural language customer inquiries into a deterministic, schema-compliant JSON format.

## 1. Environment Setup & Dependency Installation

In [ ]:
!pip install -q transformers==4.46.3 peft==0.13.2 trl==0.12.1 datasets accelerate bitsandbytes matplotlib

In [ ]:
import torch
import os
import json
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import Dataset
from trl import SFTTrainer

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Load Base Model & Configure LoRA Adapter

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

# LoRA Configuration (r=16, alpha=32, all linear projection targets)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. Dataset Preparation (Train & Held-Out Test)

In [ ]:
# Clone repo or load dataset files
!git clone https://github.com/Advik-harsha/applied-llm-systems.git || true

train_path = "applied-llm-systems/task1_finetuning/dataset/train.jsonl"
test_path = "applied-llm-systems/task1_finetuning/dataset/test.jsonl"

with open(train_path, "r") as f:
    train_data = [json.loads(line) for line in f]
with open(test_path, "r") as f:
    test_data = [json.loads(line) for line in f]

print(f"Loaded {len(train_data)} training samples and {len(test_data)} test samples.")

# Format chat messages into text
def format_record(rec):
    return {"text": tokenizer.apply_chat_template(rec["messages"], tokenize=False)}

train_dataset = Dataset.from_list([format_record(r) for r in train_data])
test_dataset = Dataset.from_list([format_record(r) for r in test_data])
print("Sample Formatted Input:")
print(train_dataset[0]["text"][:350])

## 4. Model Training with SFTTrainer

In [ ]:
OUTPUT_DIR = "./qwen2.5-0.5b-intent-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    optim="adamw_torch",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=training_args
)

print("Starting training on GPU...")
train_result = trainer.train()

# Save trained adapter
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Training finished. Model saved to {OUTPUT_DIR}")

## 5. Loss Curve Visualization

In [ ]:
log_history = trainer.state.log_history
loss_steps = [x["step"] for x in log_history if "loss" in x]
losses = [x["loss"] for x in log_history if "loss" in x]

plt.figure(figsize=(8, 4.5))
plt.plot(loss_steps, losses, marker='o', color='#1A365D', linewidth=2, label='Training Loss')
plt.title("Qwen2.5-0.5B LoRA Fine-Tuning Loss Curve", fontsize=12, fontweight='bold')
plt.xlabel("Training Steps", fontsize=10)
plt.ylabel("Cross-Entropy Loss", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.savefig("loss_curve.png", dpi=300, bbox_inches='tight')
plt.show()

## 6. Live Inference & Qualitative Output Comparison

In [ ]:
test_prompt = "Hey, I ordered the Sony WH-1000XM5 headphones with order number ORD-90182 last Tuesday. However, the left earcup makes a crackling static noise. Please send me a replacement unit right away."

messages = [
    {"role": "system", "content": "You are an AI assistant for e-commerce customer support intent parsing. Extract the customer's request into strict JSON conforming to the schema. Do not include any conversational filler or explanation."},
    {"role": "user", "content": test_prompt}
]

prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_tokens = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=False)

response_text = tokenizer.decode(output_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== Fine-Tuned Model Live Generation ===")
print(response_text)